In [26]:
from torch_geometric.datasets import KarateClub
import yaml

dataset = KarateClub()
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.

Dataset: KarateClub():
Number of graphs: 1
Number of features: 34
Number of classes: 4


In [27]:
# Load best config from wandb sweep
# path = "./wandb/sweep-df5mbgsm/config-cqvha3fh.yaml" # (GCN) put best config path here 
# path = "./wandb/sweep-2sl6k1ms/config-eouqxlrp.yaml" # (GAT) put best config path here
path = "./wandb/sweep-utsldekv/config-uh7mds6z.yaml" # (GraphSAGE) put best config path here
with open(path, "r", encoding="utf-8") as f:
    raw_config = yaml.safe_load(f)

# W&B YAML stores each parameter as {value: ...}.
best_config = {
    key: entry["value"]
    for key, entry in raw_config.items()
    if isinstance(entry, dict) and "value" in entry
}

def config_kwargs(prefix):
    return {
        key[len(prefix):]: value # round(value, 3)
        for key, value in best_config.items()
        if key.startswith(prefix)
    }

In [28]:
best_config

{'models.GraphSAGE.aggr': ['mean', 'sum', 'std'],
 'models.GraphSAGE.dropout': 0.26520107035819795,
 'models.GraphSAGE.hidden_channels': 64,
 'models.GraphSAGE.num_layers': 4,
 'optimizer.Adam.lr': 0.0027654460988353873,
 'optimizer.Adam.weight_decay': 7.940125482019007e-06,
 'training.gradient_clip_val': 1.0258849719389282,
 'wandb_project': 'karate-club-gcn-sweep'}

In [29]:
import torch
import torch.nn.functional as F

from utils.model import GCN, GAT, GraphSAGE

import random

def seed_everything(seed):
    """ Set random seeds for reproducibility.
    Args:
        seed (int): Random seed.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
# model_kwargs = config_kwargs("models.GCN.")
# model = GCN(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
# model_kwargs = config_kwargs("models.GAT.")
# model = GAT(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
model_kwargs = config_kwargs("models.GraphSAGE.")
model = GraphSAGE(in_channels=dataset.num_features,
            num_classes=dataset.num_classes,
            **model_kwargs)

print("Model config:", model_kwargs)
print(model)

Model config: {'aggr': ['mean', 'sum', 'std'], 'dropout': 0.26520107035819795, 'hidden_channels': 64, 'num_layers': 4}
GraphSAGE(
  (convs): ModuleList(
    (0): SAGEConv(34, 64, aggr=['mean', 'sum', 'std'])
    (1-3): 3 x SAGEConv(64, 64, aggr=['mean', 'sum', 'std'])
  )
  (classifier): Linear(in_features=64, out_features=4, bias=True)
)


In [30]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer_kwargs = config_kwargs("optimizer.Adam.")
# PyYAML may parse scientific notation as a string.
optimizer_kwargs = {key: float(value) for key, value in optimizer_kwargs.items()}
optimizer = torch.optim.Adam(model.parameters(), **optimizer_kwargs)
gradient_clip_val = float(best_config["training.gradient_clip_val"])
print("Adam config:", optimizer_kwargs)
print("Gradient clip:", gradient_clip_val)

def train(data):
    model.train()
    optimizer.zero_grad()  # Clear gradients.
    out = model(data.x, data.edge_index)  # Perform a single forward pass.
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
    loss.backward()  # Derive gradients.
    if gradient_clip_val > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_val)
    optimizer.step()  # Update parameters based on gradients.
    return loss

loss_history = []
for epoch in range(500):
    loss = train(data)
    loss_history.append(loss.item())
    if epoch % 10 == 0:
        print(f"epoch={epoch}, loss={loss.item():.4f}")

Adam config: {'lr': 0.0027654460988353873, 'weight_decay': 7.940125482019007e-06}
Gradient clip: 1.0258849719389282
epoch=0, loss=1.5506
epoch=10, loss=0.8086
epoch=20, loss=0.0675
epoch=30, loss=0.0004
epoch=40, loss=0.0006
epoch=50, loss=0.0000
epoch=60, loss=0.0000
epoch=70, loss=0.0000
epoch=80, loss=0.0036
epoch=90, loss=0.0000
epoch=100, loss=0.0000
epoch=110, loss=0.0000
epoch=120, loss=0.0000
epoch=130, loss=0.0000
epoch=140, loss=0.0000
epoch=150, loss=0.0000
epoch=160, loss=0.0029
epoch=170, loss=0.0000
epoch=180, loss=0.0000
epoch=190, loss=0.0000
epoch=200, loss=0.0000
epoch=210, loss=0.0000
epoch=220, loss=0.0000
epoch=230, loss=0.0006
epoch=240, loss=0.0000
epoch=250, loss=0.0000
epoch=260, loss=0.0000
epoch=270, loss=0.0000
epoch=280, loss=0.0000
epoch=290, loss=0.0000
epoch=300, loss=0.0000
epoch=310, loss=0.0000
epoch=320, loss=0.0000
epoch=330, loss=0.0000
epoch=340, loss=0.0000
epoch=350, loss=0.0000
epoch=360, loss=0.0000
epoch=370, loss=0.0006
epoch=380, loss=0.000

In [ ]:
# import numpy as np

# np.save("./graphsage_train_loss_hpt.npy", np.array(loss_history))

In [31]:
model.eval()
logits = model(data.x, data.edge_index)
test_mask = ~data.train_mask
test_loss = F.cross_entropy(logits[test_mask], data.y[test_mask]).item()
test_accuracy = (
        F.softmax(logits[test_mask],dim=1).argmax(dim=1) == data.y[test_mask]
    ).float().mean().item()

print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  21.575597763061523
test accuracy:  0.6000000238418579


In [17]:
# 예측 값
F.softmax(logits[test_mask],dim=1).argmax(dim=1)

tensor([1, 0, 1, 3, 3, 1, 0, 3, 1, 1, 1, 0, 0, 3, 1, 0, 0, 0, 1, 0, 2, 2, 0, 2,
        0, 0, 0, 2, 0, 0])

In [18]:
# 실제 값
data.y[test_mask]

tensor([1, 1, 1, 3, 3, 1, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0, 2, 0, 0,
        2, 0, 0, 2, 0, 0])